# Working with Text Data

1. **Bag-of-words/TF-IDF**
2. **A clean sklearn text pipeline**
3. **How to inspect what the model learned**
4. **Embeddings**

## 1. Core idea

Text must be turned into numbers before a model can use it.

Two classic approaches:

- CountVectorizer: represent each document by token counts
- TfidfVectorizer: represent each document by weighted token importance
- Embeddings: represent each text as a dense semantic vector

Default baseline:
`TfidfVectorizer + LogisticRegression`

In [21]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

## 2. A small example dataset

We use a tiny sentiment-style dataset just to illustrate the mechanics. In real work, this could also be survey open-ended responses, interview snippets, or social-media posts.

In [22]:
texts = ["I love this product it is simple and helpful",
         "This service is excellent and easy to use",
         "Very clear interface and smooth experience",
         "The update made everything faster and better",
         "I am happy with the results and the design",
         "Great support and very friendly communication",
         "This is confusing and frustrating",
         "The app keeps crashing and wasting my time",
         "Terrible experience I do not trust it",
         "The design is messy and hard to understand",
         "I am disappointed and the service feels unreliable",
         "Bad support and a very annoying process"]

y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

df = pd.DataFrame({"text": texts, "label": y})

## 3. CountVectorizer: bag-of-words

Bag-of-words ignores word order and records token counts.

This is simple, sparse, and still surprisingly strong for many classification tasks.

In [23]:
count_vec = CountVectorizer()

X_counts = count_vec.fit_transform(texts)

print("shape:", X_counts.shape)
print("Vocabulary size:", len(count_vec.vocabulary_))
print("First 15 tokens:", list(count_vec.vocabulary_.keys())[:15])

pd.DataFrame(
    X_counts.toarray(),
    columns=count_vec.get_feature_names_out()
).head()

shape: (12, 54)
Vocabulary size: 54
First 15 tokens: ['love', 'this', 'product', 'it', 'is', 'simple', 'and', 'helpful', 'service', 'excellent', 'easy', 'to', 'use', 'very', 'clear']


,am,and,annoying,app,bad,better,clear,communication,confusing,crashing,...,time,to,trust,understand,unreliable,update,use,very,wasting,with
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0
2,0,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


## 4. TF-IDF

Raw counts can overemphasize common words. TF-IDF downweights terms that appear everywhere and upweights terms that are more informative

In [24]:
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(texts)

print("Shape:", X_tfidf.shape)
print("Sparse matrix type:", type(X_tfidf))
pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vec.get_feature_names_out()
).round(2).head()

X_tfidf

Shape: (12, 54)
Sparse matrix type: <class 'scipy.sparse._csr.csr_matrix'>


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 82 stored elements and shape (12, 54)>

In [25]:
row = X_tfidf[0].toarray().ravel()
top_idx = row.argmax()
feature_names = tfidf_vec.get_feature_names_out()
print("word:", feature_names[top_idx])
print("tf-idf:", row[top_idx])

word: helpful
tf-idf: 0.41105995357076414


## 5. The standard text classification pipeline

vectorizer -> classifier

In [26]:
X_train, X_test, y_train, y_test = train_test_split(texts, y, test_size=0.25, random_state = 42, stratify=y)

pipe = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(max_iter=2000)
)

pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)

print("Prediction:", pred)
print("\nClassification report:")
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))

Prediction: [1 1 1]

Classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3

Confusion matrix:
[[0 2]
 [0 1]]


## 6. Cross-validation on the whole workflow

The whole text pipeline should be cross-validated, not just the classifier.

That means tokenization, vocabulary building, and TF-IDF fitting are all learned inside each fold.

In [27]:
cv_scores = cross_val_score(pipe, texts, y, cv=4)
print("CV scores:", cv_scores)
print("Mean CV score:", cv_scores.mean().round(3))

CV scores: [0.33333333 0.33333333 0.33333333 0.33333333]
Mean CV score: 0.333


## 7. Inspecting what the model learned

For linear models, we can inspect high-weight features

In [28]:
vectorizer = pipe.named_steps["tfidfvectorizer"]
model = pipe.named_steps["logisticregression"]

feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

top_positive_idx = np.argsort(coefs)[-10:][::-1]
top_negative_idx = np.argsort(coefs)[:10]

top_positive = pd.DataFrame({
    "feature": feature_names[top_positive_idx],
    "coef": coefs[top_positive_idx]
})

top_negative = pd.DataFrame({
    "feature": feature_names[top_negative_idx],
    "coef": coefs[top_negative_idx]
})


print("Top positive features")
display(top_positive)

print("Top negative features")
display(top_negative)




Top positive features


,feature,coef
0,friendly,0.128403
1,great support,0.128403
2,friendly communication,0.128403
3,great,0.128403
4,communication,0.128403
5,very friendly,0.128403
6,very clear,0.119251
7,clear interface,0.119251
8,smooth,0.119251
9,and smooth,0.119251


Top negative features


,feature,coef
0,is confusing,-0.172212
1,confusing and,-0.172212
2,and frustrating,-0.172212
3,confusing,-0.172212
4,this is,-0.172212
5,frustrating,-0.172212
6,bad support,-0.159484
7,bad,-0.159484
8,annoying process,-0.159484
9,process,-0.159484


## 8. What matters most in practice

For classic text ML, the main decisions are usually:
- What text unit? document/sentence/response
- What vectorizer? counts vs TF-IDF
- What token pattern? words, character n-grams, lowercasing
- What n-grams? unigrams only, or include bigrams
- What model? linear models are often very strong baselines

A good first baseline is often:

```python
make_pipeline(
    TfidfVectorizer(ngram_range(1, 2)),
    LogisticRegression(max_iter=2000)
)
```

## 9. Embeddings

Embeddings give each text a dense vector that captures more semantic information.
They are especially useful for:

- semantic similarity
- clustering
- retrieval
- downstream classification with richer representations


## 10. When to use what

### Use bag-of-words / TF-IDF when:
- you want a strong, fast baseline
- interpretability matters
- dataset size is modest
- you want a transparent sklearn workflow

### Use embeddings when:
- semantics matter more
- you want similarity / clustering / retrieval
- text is short and varied
- you plan to move toward modern NLP / LLM workflows

## 11. For open-ended survey responses

A sensible order would be:

1. Build a **TF-IDF + LogisticRegression** baseline
2. Inspect important features
3. Try **embedding-based clustering or similarity**
4. Think explicitly about **validity and bias**

That last step matters a lot in social-science text work:
- Are labels conceptually valid?
- Are categories stable?
- Could the model be learning style rather than meaning?
- Is performance uneven across subgroups?

## 12. Takeaway

The key workflow is simple:

**text -> vectorizer -> model**

The most reusable baseline is:

**`TfidfVectorizer + LogisticRegression`**

And the most important update is:

**keep the classic pipeline, but add an embedding perspective when semantics become central.**
